In [ ]:
import sys
import gc
from time import perf_counter
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# from torch_cluster import radius_graph

from typing import Optional

import cProfile
import pstats
from memory_profiler import memory_usage
from numba import njit, prange
from time import perf_counter

from gudhi import CubicalComplex
import cripser
import dionysus as dion

from topo.persistence_1d import find_extrema_in_timeseries, compute_1d_sublevel_persistence, compute_batch_sublevel_persistence, \
    StreamingSublevelPersistence, BatchStreamingSublevelPersistence
from utils import DataUtils, MemoryUtils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
"💾 memory cleanup"
MemoryUtils.clear_jupyter_state()

MemoryUtils.clear_runtime_memory()

MemoryUtils.clear_compilation_cache()


In [ ]:
"computations"
n_points     = 100
point_cloud  = DataUtils.generate_torus_point_cloud(n_points, R_major=1.0, r_minor=0.10, noise=0.005, rand_seed=4, device=device)

start_time = perf_counter()

D        = torch.cdist(point_cloud, point_cloud)
max_dist = D.max().item()
D        = D / max_dist # normalize to [0,1], better numerical stability

# ~~~~~~~~~~~~~~~~~~~~~
src, dst = torch.triu_indices(n_points, n_points, offset=1, device=device)
weights  = D[src, dst]
perm     = torch.argsort(weights)
src      = src[perm]
dst      = dst[perm]
weights  = weights[perm]
uf = UnionFind(n_points)

B0_arr = []
B1_arr = []

E = 0
for i in range(len(weights)):
    u = int(src[i])
    v = int(dst[i])

    uf.union(u, v)
    if uf.find(u) != uf.find(v):
        print(f"merge at eps={weights[i].item():.3f}")
    E += 1
    B0 = uf.num_components()
    B1 = E - n_points + B0

    if i % 500 == 0:
        print(
            f"step={i}/{len(weights)} | "
            f"eps={weights[i].item():.4f} | "
            f"E={E} | B0={B0} | B1={B1}")
    B0_arr.append(B0)
    B1_arr.append(B1)

# ~~~~~~~~~~~~~~~~~~~~~
# B0_arr, B1_arr = [], []
# betti_obj = BettiComputer(n_triangles=0, n_edges=0, n_vertices=n_points, device=device)

# counter= 0
# eps    = 0
# step   = 0.05

# while eps <= max_dist:
#     adj         = adjacency_from_distance_matrix(D, eps)
#     n_edges     = count_edges(adj)
#     n_triangles = count_triangles_from_adjacency(adj)
    
#     edge_index = radius_graph(point_cloud, r=eps, loop=False)
#     # B0 = betti_obj.compute_betti_0(adj, tolerance=1e-5)
#     B0 = betti_obj.compute_betti_0_edge_index(edge_index, n_vertices=n_points)
#     E  = edge_index.shape[1]
#     B1 = betti_obj.compute_betti_1(V=n_points, E=E, B0=B0)
#     B0_arr.append(B0)
#     B1_arr.append(B1)

#     density = E / (n_points * (n_points - 1) // 2)

#     if (counter + 1) % 5 == 0:
#         print(f"eps={eps:.2f}/{max_dist:.2f} | "
#             f"edges={n_edges}/{n_points*(n_points-1)//2} | "
#             f"triangles={n_triangles}/{math.comb(n_points, 3)} | "
#             f"Betti: B0={B0}, B1={B1} | "
#             f"Density: {density:.4f}")
#     eps += step
#     counter += 1
# # ~~~~~~~~~~~~~~~~~~~~~

end_time = perf_counter()
print(f"⚠️ Operations completed in {end_time - start_time:.4f} seconds")

plt.figure()
plt.plot(np.arange(len(B0_arr)) * step, np.log(B0_arr), label='log(B0)')
plt.plot(np.arange(len(B1_arr)) * step, np.log(B1_arr), label='log(B1)')
plt.legend()
plt.show()

# ===============
plt.scatter(point_cloud[:, 0].cpu(), point_cloud[:, 1].cpu(), s=2)
plt.gca().set_aspect('equal')
plt.show()
plt.figure()
sns.heatmap(D.cpu(), cmap='viridis')
plt.title("Distance matrix")


In [ ]:
"✅📊 Load real datasets"
df       = pd.read_csv("public_datasets/2D/tabular/nvidia_dataset/NVidia_stock_history.csv")
y_vals_t = torch.tensor(df["Close"].values, dtype=torch.float32)#, device=device)

y_vals_flat = y_vals_t.repeat(1, 200).flatten()
y_vals = F.avg_pool1d(y_vals_flat.unsqueeze(0).unsqueeze(0), kernel_size=50, stride=1,).squeeze()
print(y_vals.shape, y_vals_t.shape)

# df     = pd.read_csv("../public_datasets/2D/tabular/longterm_weather/longterm_weather.csv")
# y_vals = torch.tensor(df["rh"].values, dtype=torch.float32)#, device=device)
y_new   = y_vals[:200]
y_new2  = y_vals[:200]
y_vals.shape
num_timesteps = y_vals.shape[0]
num_timesteps_new = num_timesteps
end_point = num_timesteps - 1


In [ ]:
"✅ setup function-based 1D array"
matplotlib.rcParams['text.usetex'] = False

start_point   = 0
num_timesteps = 10_000
end_point     = num_timesteps//1
# x_vals = torch.linspace(start_point, end_point, num_timesteps)
y_vals        = DataUtils.make_yvals_1d(num_timesteps, start_point, end_point)

num_timesteps_new = 100_000
start_point_new   = end_point + 1
end_point_new     = start_point_new+num_timesteps_new
y_new = DataUtils.make_yvals_1d(num_timesteps_new, start_point_new, end_point_new)

num_timesteps_new2 = 100_000
start_point_new2   = end_point_new + 1
end_point_new2     = start_point_new2 + num_timesteps_new2
y_new2 = DataUtils.make_yvals_1d(num_timesteps_new2, start_point_new2, end_point_new2)


In [ ]:
"🪑 [1D] benchmarks"

# # ===== 🪷 Gudhi =====
# start_gudhi = perf_counter()
# cc = CubicalComplex(dimensions=y_vals.shape, top_dimensional_cells=y_vals.numpy().flatten())
# cc.compute_persistence(homology_coeff_field=2) # default is 11, but 2 is faster for Z/2Z coefficients
# # pairs = cc.persistence()  # (dimension, (birth, death))

# # gudhi.plot_persistence_diagram(cc.persistence())
# # gudhi.plot_persistence_barcode(cc.persistence())

# # plt.figure()
# # plt.plot(x_vals.numpy(), y_vals.numpy())
# # plt.show()

# endtime_gudhi = perf_counter()
# print(f"🧮 [🪷 gudhi] persistence of {num_timesteps} y vals in {endtime_gudhi - start_gudhi:.4f} s")

# def bench_gudhi():
#     cc = CubicalComplex(dimensions=y_vals.shape, top_dimensional_cells=y_vals.numpy().flatten())
#     return cc.compute_persistence(homology_coeff_field=2) # default is 11, but 2 is faster for Z/2Z coefficients
# peak_mem_gudhi = max(memory_usage(bench_gudhi))
# print(f"💾 [🪷 gudhi] Peak RAM: {peak_mem_gudhi:.2f} MiB")

# start_gudhi2 = perf_counter()
# cc2 = CubicalComplex(dimensions=y_new.shape, top_dimensional_cells=y_new.numpy().flatten())
# cc2.compute_persistence(homology_coeff_field=2)
# # pairs2 = cc2.persistence()
# endtime_gudhi2 = perf_counter()
# print(f"⛓️‍💥 [🪷 gudhi] stream persistence of {num_timesteps_new} y vals in {endtime_gudhi2 - start_gudhi2:.4f} s")


# ===== ⚡️ cripser =====
arr             = y_vals.numpy()  # keep as 1D array, no need to flatten
start_cripser   = perf_counter()

# prof = cProfile.Profile()
# prof.enable()
ph              = cripser.compute_ph(arr, maxdim=0)
# prof.disable()
# stats = pstats.Stats(prof)
# stats.sort_stats("cumtime")
# stats.print_stats(30)

endtime_cripser = perf_counter()
print(f"🧮 [⚡️ cripser] persistence of {num_timesteps} y vals in {endtime_cripser - start_cripser:.4f} s")

# def bench_cripser():
#     return cripser.compute_ph(arr, maxdim=0)
# peak_mem_cripser = max(memory_usage(bench_cripser))

peak_mem_cripser = max(memory_usage(lambda: cripser.compute_ph(arr, maxdim=0)))
print(f"💾 [⚡️ cripser] Peak RAM: {peak_mem_cripser:.2f} MiB")


# ===== 🦕 Dionysus =====
# start_dionysus = perf_counter()
# f    = dion.fill_freudenthal(y_vals.numpy())
# m    = dion.homology_persistence(f)
# # dgms = dion.init_diagrams(m, f)
# endtime_dionysus = perf_counter()
# print(f"🧮 [🦕 dionysus] persistence of {num_timesteps} y vals in {endtime_dionysus - start_dionysus:.4f} s")

# def bench_dionysus():
#     return dion.homology_persistence(dion.fill_freudenthal(y_vals.numpy()))
# peak_mem_dionysus = max(memory_usage(bench_dionysus))
# print(f"💾 [🦕 dionysus] Peak RAM: {peak_mem_dionysus:.2f} MiB")


In [ ]:
"⛲️ new water sublevel, no numba"
from persistence_lake import compute_1d_sublevel_persistence_lake, PersistenceLake1D, OnlineSublevelPersistenceLake
from old_1d_persistence import Sublevel1D, IncrementalSublevel

# 1. Extract keypoints for your baseline history (y_vals)
sublevel_baseline = Sublevel1D(x=y_vals)
base_keypoint_idx, base_keypoint_types = sublevel_baseline.find_extrema_in_timeseries()

batch_pairs = compute_1d_sublevel_persistence_lake(y_vals, base_keypoint_idx, base_keypoint_types)

print(f"Total pairs found in full batch: {len(batch_pairs)}")
for b, d in batch_pairs:
    print(f"Batch Pair: ({b}, {d})")

# 2. INITIALIZE the online persistence engine with your baseline history
online_tda = OnlineSublevelPersistenceLake(y_vals, base_keypoint_idx, base_keypoint_types)

# 3. Extract keypoints for your incoming streaming chunk (y_new)
sublevel_stream = Sublevel1D(x=y_new)
stream_keypoint_idx, stream_keypoint_types = sublevel_stream.find_extrema_in_timeseries()

# 4. Append the streaming chunk to process incremental changes
# This outputs ONLY the new pairs closed by the arrival of y_new
birth_death_pairs_list = online_tda.append_stream_data(
    new_x_chunk=y_new, 
    stream_keypoint_idx=stream_keypoint_idx, 
    stream_keypoint_types=stream_keypoint_types)

# 5. PRINT RESULTS (Using your exact style)
# Note: online_tda.history_values is now automatically the unified (y_vals + y_new) tensor

print("--- Persistence Pairs: (birth idx, death idx) ---")
for birth_time_idx, death_time_idx in birth_death_pairs_list:
    print(f"({birth_time_idx}, {death_time_idx})")

print("\n--- y values: (birth, death; lifetime) ---")
for birth_idx, death_idx in birth_death_pairs_list:
    birth_val = online_tda.history_values[birth_idx].item()
    death_val = online_tda.history_values[death_idx].item()
    print(f"({birth_val:2.2f}, {death_val:2.2f}; {death_val - birth_val:2.2f})")


In [ ]:
"⏰ [1D] gpu + numba"

# 1. INITIAL BASELINE (Compute & View Pairs)
torch.cuda.synchronize() if device.type == 'cuda' else None
starttime_gpu = perf_counter()

# prof  = cProfile.Profile()
# prof.enable()
keypoint_idx, keypoint_types = find_extrema_in_timeseries(y_vals, device=y_vals.device)
birth_death_pairs_list       = compute_1d_sublevel_persistence(y_vals, keypoint_idx, keypoint_types)
# prof.disable()
# stats = pstats.Stats(prof)
# stats.sort_stats("cumtime")
# stats.print_stats(30)

torch.cuda.synchronize() if device.type == 'cuda' else None
endtime_gpu   = perf_counter()
print(f"🧮 [😎 ours] persistence of {num_timesteps} yvals in {endtime_gpu - starttime_gpu:.4f}s")

# RAM
def bench_gpu():
    kp_idx, kp_types = find_extrema_in_timeseries(y_vals, device=y_vals.device)
    return compute_1d_sublevel_persistence(y_vals, kp_idx, kp_types)
peak_mem = max(memory_usage(bench_gpu))
print(f"💾 Peak RAM: {peak_mem:.2f} MiB")

# GPU VRAM (separate, additional cost)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats() if device.type == 'cuda' else None
kp_idx, kp_types = find_extrema_in_timeseries(y_vals, device=y_vals.device)
compute_1d_sublevel_persistence(y_vals, kp_idx, kp_types)
torch.cuda.synchronize() if device.type == 'cuda' else None
print(f"💾 Peak VRAM: {torch.cuda.max_memory_allocated()/1024**2:.2f} MiB")

# 2. INITIALIZE STREAMING PIPELINE (Allocate Max Capacity Upfront Once)
total_capacity = len(y_vals) + len(y_new) + len(y_new2) # set total capacity to safely fit everything
online_tda     = StreamingSublevelPersistence(y_vals, keypoint_idx, keypoint_types, max_capacity=total_capacity)

# 3. STREAM CHUNK 1 (y_new)
stream_keypoint_idx, stream_keypoint_types = find_extrema_in_timeseries(y_new, device=y_new.device)

# Guard condition directly inside your notebook runner cell
if stream_keypoint_idx.numel() == 0:
    print(f"ℹ️ stream chunk 1 has no topology features (flat line). Skipping state update.")
    new_birth_death_pairs_list = []
else:
    new_birth_death_pairs_list = online_tda.update_filtration(
        new_y_chunk=y_new,
        stream_keypoint_idx=stream_keypoint_idx,
        stream_keypoint_types=stream_keypoint_types)
endtime_gpu2 = perf_counter()
print(f"⛓️‍💥 stream chunk 1 persistence of {len(y_new)} yvals in {endtime_gpu2 - endtime_gpu:.4f}s")

# 4. STREAM CHUNK 2 (y_new2)
stream_keypoint_idx2, stream_keypoint_types2 = find_extrema_in_timeseries(y_new2, device=y_new2.device)

# Guard condition directly inside your notebook runner cell
if stream_keypoint_idx2.numel() == 0:
    print(f"ℹ️ stream chunk 2 has no topology features (flat line). Skipping state update.")
    new_birth_death_pairs_list2 = []
else:
    new_birth_death_pairs_list2 = online_tda.update_filtration(
        new_y_chunk=y_new2,
        stream_keypoint_idx=stream_keypoint_idx2,
        stream_keypoint_types=stream_keypoint_types2)
endtime_gpu3 = perf_counter()
print(f"⛓️‍💥 stream chunk 2 persistence of {len(y_new2)} yvals in {endtime_gpu3 - endtime_gpu2:.4f}s")


In [ ]:
"🍌 [1D] banana tree implementation"
from topo.banana_tree_algo import DynamicMergeTreeCEH

starttime_bananatree = perf_counter()
kp_idx, kp_types     = find_extrema_in_timeseries(y_vals, device)
m = DynamicMergeTreeCEH(max_capacity=len(y_vals))
indices = kp_idx.cpu().numpy()
heights = y_vals[kp_idx].cpu().numpy()
types   = kp_types.cpu().numpy()
discovered_pairs   = m.update_stream(indices, heights, types)
endtime_bananatree = perf_counter()
print(f"🧮 [🍌 bananatree] persistence of {num_timesteps} yvals in {endtime_bananatree - starttime_bananatree:.4f}s")

def bench_banana():
    kp_idx, kp_types = find_extrema_in_timeseries(y_vals, device)
    m       = DynamicMergeTreeCEH(max_capacity=len(y_vals))
    indices = kp_idx.cpu().numpy()
    heights = y_vals[kp_idx].cpu().numpy()
    types   = kp_types.cpu().numpy()
    discovered_pairs = m.update_stream(indices, heights, types)
    return discovered_pairs
peak_mem_banana = max(memory_usage(bench_banana))
print(f"💾 Peak RAM: {peak_mem_banana:.2f} MiB")


In [ ]:
"🧩 [1D] BATCHING (no streaming)"

y_batch = torch.stack([y_vals, 1.2 * y_vals, -0.8 * y_vals, -0.4+2.1 * y_vals, 0.2-0.1 * y_vals])

starttime_batch = perf_counter()

# 2. Extract unique keypoints into raw Python lists (Lengths can safely mismatch!)
idx_list, types_list = [], []
for i in range(y_batch.shape[0]):
    idx, types = find_extrema_in_timeseries(y_batch[i], device=y_batch.device)
    idx_list.append(idx)
    types_list.append(types)

# 3. Run the updated auto-padded batch calculator
all_batch_pairs = compute_batch_sublevel_persistence(y_batch, idx_list, types_list)

endtime_batch = perf_counter()
print(f"🧮 persistence of {num_timesteps} yvals in {endtime_batch - starttime_batch:.4f}s")

# for i, single_series_pairs in enumerate(all_batch_pairs):
#     print(f"\n--- Time Series #{i} Persistence Pairs ---")
#     for b_idx, d_idx in single_series_pairs[:5]:  # print first 5 pairs as a test
#         birth_val = y_batch[i, b_idx].item()
#         death_val = y_batch[i, d_idx].item()
#         print(f"Indices: ({b_idx}, {d_idx}) -> Values: ({birth_val:2.2f}, {death_val:2.2f})")


In [ ]:
"🥋 [1D] MASTER (batching + streaming)"

starttime_master = perf_counter()

# --- STEP 1: Slice your base data into a "Baseline" block and a "Streaming Chunk" block ---
baseline_cutoff = int(len(y_vals) * 0.8)

y_vals_baseline = y_vals[:baseline_cutoff]
y_vals_chunk    = y_vals[baseline_cutoff:]

# --- STEP 2: Build the Initial Baseline 2D Matrix (N = 5) ---
y_batch = torch.stack([
    y_vals_baseline, 
    1.2 * y_vals_baseline, 
    -0.8 * y_vals_baseline, 
    -0.4 + 2.1 * y_vals_baseline, 
    0.2 - 0.1 * y_vals_baseline])

# --- STEP 3: Build your incoming streaming 2D Matrix (N = 5) ---
y_chunk_matrix = torch.stack([
    y_vals_chunk, 
    1.2 * y_vals_chunk, 
    -0.8 * y_vals_chunk, 
    -0.4 + 2.1 * y_vals_chunk, 
    0.2 - 0.1 * y_vals_chunk])

# --- STEP 4: Run the pipeline ---
idx_list, types_list = [], []
for i in range(y_batch.shape[0]):
    idx, types = find_extrema_in_timeseries(y_batch[i], device=y_batch.device)
    idx_list.append(idx)
    types_list.append(types)

# Spin up the global 2D tracking system
batch_online_tda = BatchStreamingSublevelPersistence(y_batch, idx_list, types_list)

# Extract chunk keypoints
new_idx_list, new_types_list = [], []
for i in range(y_chunk_matrix.shape[0]):
    idx, types = find_extrema_in_timeseries(y_chunk_matrix[i], device=y_chunk_matrix.device)
    new_idx_list.append(idx)
    new_types_list.append(types)

# Process the new chunk instantaneously across all 5 channels!
new_batch_pairs = batch_online_tda.append_batch_stream(y_chunk_matrix, new_idx_list, new_types_list)

endtime_master = perf_counter()
print(f"🧮 Persistence of {len(y_vals_chunk)} streaming yvals processed across {y_batch.shape[0]} channels in {endtime_master - starttime_master:.4f}s")

# # --- STEP 5: Verification Printout (Sanity Check) ---
# print("\n🔍 Verification Lookups:")
# for i, single_series_pairs in enumerate(new_batch_pairs):
#     print(f"  Channel #{i}: Found {len(single_series_pairs)} new pairs.")
#     if len(single_series_pairs) > 0:
#         b_idx, d_idx = single_series_pairs[0]  # Check first pair discovered
#         birth_val = batch_online_tda.history_values[i, b_idx]
#         death_val = batch_online_tda.history_values[i, d_idx]
#         print(f"    └─ Sample Pair -> Indices: ({b_idx}, {d_idx}) | Values: ({birth_val:2.3f}, {death_val:2.3f})")



In [ ]:
"🥋 [1D] ADAPTIVE MULTI-STREAM BATCH RUNNER"

# Slicing the remainder of y_vals into 3 distinct chronological frames
chunk_size = len(y_vals[baseline_cutoff:]) // 3

y_vals_stream_1 = y_vals[baseline_cutoff : baseline_cutoff + chunk_size]
y_vals_stream_2 = y_vals[baseline_cutoff + chunk_size : baseline_cutoff + (2 * chunk_size)]
y_vals_stream_3 = y_vals[baseline_cutoff + (2 * chunk_size) :]

# Group the raw streams into the list
incoming_raw_streams = [y_vals_stream_1, y_vals_stream_2, y_vals_stream_3]

# =====================================================================
# STEP 1: Define Your Stream Collection (Can hold 0, 1, or more tensors)
# =====================================================================
incoming_raw_streams = [y_vals_stream_1, y_vals_stream_2, y_vals_stream_3] 

# =====================================================================
# STEP 2: Initialize with Base Data (Always Executed)
# =====================================================================
idx_list, types_list = [], []
for i in range(y_batch.shape[0]):
    idx, types = find_extrema_in_timeseries(y_batch[i], device=y_batch.device)
    idx_list.append(idx)
    types_list.append(types)

# Spin up the tracker
batch_online_tda = BatchStreamingSublevelPersistence(y_batch, idx_list, types_list, max_capacity=1_000_000)


# =====================================================================
# STEP 3: Zero-Allocation Adaptive Streaming Loop
# =====================================================================
if not incoming_raw_streams:
    print("ℹ️ No incoming streams provided. Baseline state initialized and preserved.")
else:
    print(f"🚀 Processing {len(incoming_raw_streams)} incoming sequential stream steps across {y_batch.shape[0]} channels...\n")

    for stream_idx, raw_chunk in enumerate(incoming_raw_streams, start=1):
        # Defend against empty streaming ticks
        if raw_chunk is None or len(raw_chunk) == 0:
            print(f"⚠️ [Stream #{stream_idx}] Empty tensor chunk received. Skipping.")
            continue
            
        starttime_chunk = perf_counter()
        
        # 1. Re-apply your custom transformations to construct the 2D chunk matrix
        y_chunk_matrix = torch.stack([
            raw_chunk, 
            1.2 * raw_chunk, 
            -0.8 * raw_chunk, 
            -0.4 + 2.1 * raw_chunk, 
            0.2 - 0.1 * raw_chunk])
        
        # 2. Extract keypoints for this specific window frame
        chunk_idx_list, chunk_types_list = [], []
        for i in range(y_chunk_matrix.shape[0]):
            idx, types = find_extrema_in_timeseries(y_chunk_matrix[i], device=y_chunk_matrix.device)
            chunk_idx_list.append(idx)
            chunk_types_list.append(types)
            
        # 3. Append the batch matrix down the pipeline
        stream_batch_pairs = batch_online_tda.append_batch_stream(y_chunk_matrix, chunk_idx_list, chunk_types_list)
        
        endtime_chunk = perf_counter()
        print(f"📦 [Stream #{stream_idx}] Processed {y_chunk_matrix.shape[1]} timesteps in {endtime_chunk - starttime_chunk:.4f}s")
        print(f"   └─ Channel #0 found {len(stream_batch_pairs[0])} new persistent features.")

In [ ]:
"[1D] speedup comparer"
N_points = end_point
K        = len(keypoint_idx)

cripser_ops = 2*N_points*np.log2(2*N_points) + 6*N_points
our_ops     = 2*N_points + K*np.log2(K) + 4*K

print(f"{N_points} points, {K} keypoints, {K/N_points*100:.2f}% are keypoints")
print(f"#ops: Cripser={cripser_ops:.2e}, Ours={our_ops:.2e}")

theoretical_speedup = cripser_ops / our_ops
actual_speedup = (endtime_cripser - start_cripser) / (endtime_gpu - starttime_gpu)
speedup_ratio = actual_speedup/theoretical_speedup
speedup_emoji = "🚀" if actual_speedup > theoretical_speedup else "🐌"
print(f"Speedup: theoretical={theoretical_speedup:.2f}x actual={actual_speedup:.2f}x ratio={speedup_ratio:.2f} {speedup_emoji}")


In [ ]:
"📗 [2D] setup"
# arr_2d    = DataUtils.generate_random_matrix(50, 50, low=0.0, high=30)
arr_2d    = DataUtils.generate_patchy_matrix(400, 400, device=device, min_val=0.0, max_val=100, smooth_radius=2)
# arr_2d = DataUtils.generate_donut_matrix(20, 20, inner_radius=2.5, outer_radius=4.0)
arr_2d_np = arr_2d.numpy()

# start_gudhi_2d  = perf_counter()
# cc = CubicalComplex(dimensions=arr_2d_np.shape, top_dimensional_cells=arr_2d_np.flatten())
# cc.compute_persistence(homology_coeff_field=2) # default is 11, but 2 is faster for Z/2Z coefficients
# endtime_gudhi_2d= perf_counter()
# print(f"🧮 [🪷 gudhi] persistence of {arr_2d.shape[0]*arr_2d.shape[1]} y vals in {endtime_gudhi_2d - start_gudhi_2d:.4f} s")

# start_cripser_2d  = perf_counter()
# ph_2d             = cripser.compute_ph(arr_2d_np, maxdim=1)
# endtime_cripser_2d= perf_counter()
# print(f"🧮 [⚡️ cripser] persistence of {arr_2d.shape[0]*arr_2d.shape[1]} y vals in {endtime_cripser_2d - start_cripser_2d:.4f} s")

# def bench_cripser():
#     return cripser.compute_ph(arr_2d_np, maxdim=1)
# peak_cripser_mem = max(memory_usage(bench_cripser))
# print(f"💾 Peak RAM: {peak_cripser_mem:.2f} MiB")

# # Separate H0 and H1 diagrams
# h0_cripser = ph_2d[ph_2d[:, 0] == 0][:, 1:3]  # Keeps only [Birth, Death]
# h1_cripser = ph_2d[ph_2d[:, 0] == 1][:, 1:3]  # Keeps only [Birth, Death]

# # Print summary to verify profiles
# print(f"CRiPSER # H0: {len(h0_cripser)}, # H1: {len(h1_cripser)}")
# # print("First few H1 pairs:\n", h1_cripser[:3])

plt.imshow(arr_2d, aspect="auto", cmap="bwr")
plt.colorbar()
plt.show()

In [ ]:
"🪑 [2D] benchmarks"

arr_2d_np = arr_2d.detach().cpu().numpy().astype('float32')
MemoryUtils.clear_memory()

print(f"\n==================== RUNNING BASELINE BENCHMARKS (3x Trials) ====================")

# 2. Cripser Benchmark Setup
def run_cripser():
    return cripser.compute_ph(arr_2d_np, maxdim=1)

cripser_times = []
cripser_mems = []

for trial in range(3):
    MemoryUtils.clear_memory()
    start_cripser_2d = perf_counter()
    ph_2d = run_cripser()
    endtime_cripser_2d = perf_counter()
    cripser_times.append(endtime_cripser_2d - start_cripser_2d)
    
    peak_mem_cripser = max(memory_usage(run_cripser))
    cripser_mems.append(peak_mem_cripser)
    del ph_2d

print(f"🧮 [⚡️ cripser] Time:     {np.mean(cripser_times):.4f} s ± {np.std(cripser_times):.4f} s")
print(f"💾 [⚡️ cripser] Peak RAM: {np.mean(cripser_mems):.2f} MiB ± {np.std(cripser_mems):.2f} MiB")
print(f"CubicalRipser & {np.mean(cripser_times):.3f}$\\pm${np.std(cripser_times):.3f} & {np.mean(cripser_mems):.2f}$\\pm${np.std(cripser_mems):.2f} \\")

MemoryUtils.clear_memory()


# 1. GUDHI Benchmark Setup
def run_gudhi():
    cc = CubicalComplex(dimensions=arr_2d_np.shape, top_dimensional_cells=arr_2d_np.flatten())
    return cc.compute_persistence(homology_coeff_field=2)

gudhi_times = []
gudhi_mems = []

for trial in range(3):
    MemoryUtils.clear_memory()
    start_gudhi_2d = perf_counter()
    _ = run_gudhi()
    endtime_gudhi_2d = perf_counter()
    gudhi_times.append(endtime_gudhi_2d - start_gudhi_2d)
    
    peak_mem_gudhi = max(memory_usage(run_gudhi))
    gudhi_mems.append(peak_mem_gudhi)
    print(f"💾 [🪷 gudhi] Peak RAM: {peak_mem_gudhi:.2f} MiB")
    print(f"🧮 [🪷 gudhi] Time:     {endtime_gudhi_2d - start_gudhi_2d:.4f} s")

print(f"🧮 [🪷 gudhi] Time:     {np.mean(gudhi_times):.4f} s ± {np.std(gudhi_times):.4f} s")
print(f"💾 [🪷 gudhi] Peak RAM: {np.mean(gudhi_mems):.2f} MiB ± {np.std(gudhi_mems):.2f} MiB")
print(f"Gudhi & {np.mean(gudhi_times):.3f}$\\pm$ {np.std(gudhi_times):.3f} & {np.mean(gudhi_mems):.2f}$\\pm${np.std(gudhi_mems):.2f} \\")

MemoryUtils.clear_memory()

# 3. Dionysus 2 Benchmark Setup
# def run_dionysus():
#     f = dion.fill_freudenthal(arr_2d_np)
#     return dion.homology_persistence(f)

# dionysus_times = []
# dionysus_mems = []

# for trial in range(3):
#     MemoryUtils.clear_memory()
#     start_dionysus = perf_counter()
#     m = run_dionysus()
#     endtime_dionysus = perf_counter()
#     dionysus_times.append(endtime_dionysus - start_dionysus)
    
#     peak_mem_dionysus = max(memory_usage(run_dionysus))
#     dionysus_mems.append(peak_mem_dionysus)
#     del m

# MemoryUtils.clear_memory()

# print(f"🧮 [🦕 dionysus] Time:     {np.mean(dionysus_times):.4f} s ± {np.std(dionysus_times):.4f} s")
# print(f"💾 [🦕 dionysus] Peak RAM: {np.mean(dionysus_mems):.2f} MiB ± {np.std(dionysus_mems):.2f} MiB")
# print("================================================================================\n")

In [ ]:
"our method 2d"
# from skimage.segmentation import watershed
# from skimage.feature import peak_local_max
from topo.persistence_2d import compute_h0_h1_fast, run_streaming_persistence, prepare_multi_chunk_wrapper

print(f"\n==================== WAY 1: GLOBAL (3x Trials) ====================")

global_times = []
global_mems = []

def bench_2d():
    return compute_h0_h1_fast(arr_2d)

# Warmup pass to eliminate initial JIT/Numba compilation overhead from stats
_ = bench_2d()
MemoryUtils.clear_memory()

for trial in range(1):
    MemoryUtils.clear_memory()
    start_our_2d = perf_counter()
    h0, h1 = compute_h0_h1_fast(arr_2d)
    endtime_our_2d = perf_counter()
    global_times.append(endtime_our_2d - start_our_2d)
    
    peak_mem_global = max(memory_usage(bench_2d))
    global_mems.append(peak_mem_global)
    
    if trial == 2:  # Save dimensions for reference report on last trial
        h0_len, h1_len = len(h0), len(h1)
    del h0, h1

print(f"🧮 [😎 ours global] Time:     {np.mean(global_times):.4f} s ± {np.std(global_times):.4f} s")
print(f"💾 [😎 ours global] Peak RAM: {np.mean(global_mems):.2f} MiB ± {np.std(global_mems):.2f} MiB")
# print(f"   ↳ # H0 pairs: {h0_len}, # H1 pairs: {h1_len}")
# print(f"Ours & {np.mean(global_times):.3f}$\\pm${np.std(global_times):.3f} & {np.mean(global_mems):.2f}$\\pm${np.std(global_mems):.2f} \\")

MemoryUtils.clear_memory()


print(f"\n==================== WAY 2: ROW-WISE CHUNK STREAMING (3x Trials) ====================")

m_chunks = 10
stream_times = []
stream_mems = []
stream_vms = []

def bench_streaming():
    dp = prepare_multi_chunk_wrapper(arr_2d, m_chunks=m_chunks)
    return run_streaming_persistence(dp)

# Warmup pass to eliminate JIT compilation on streaming loops
_ = bench_streaming()
MemoryUtils.clear_memory()

for trial in range(2):
    MemoryUtils.clear_memory()
    
    if trial == 0:
        print(f"📦 Simulating stream arrival of {m_chunks} horizontal cuts (Verbose logging on Trial 1)...")
    
    start_streaming_total = perf_counter()
    chunks = torch.tensor_split(arr_2d, m_chunks, dim=0)
    packed_chunk_data = []

    for idx, chunk in enumerate(chunks):
        start_chunk = perf_counter()
        chunk_package = prepare_multi_chunk_wrapper(chunk, m_chunks=1)
        packed_chunk_data.append(chunk_package)
        end_chunk = perf_counter()
        if trial == 0:
            print(f"  ↳ 🚰 [Chunk {idx+1}/{m_chunks}] Footprint {chunk.shape[0]}x{chunk.shape[1]} in {end_chunk - start_chunk:.4f}s")

    if trial == 0:
        print(f"⚙️  Executing interface stitching across boundaries and compiling final diagram...")
        
    data_package = prepare_multi_chunk_wrapper(arr_2d, m_chunks=m_chunks)
    h0_stream, h1_stream = run_streaming_persistence(data_package)

    end_streaming_total = perf_counter()
    stream_times.append(end_streaming_total - start_streaming_total)
    stream_vms.append(get_vms_mib())
    
    peak_mem_stream = max(memory_usage(bench_streaming))
    stream_mems.append(peak_mem_stream)
    
    if trial == 2:
        h0_stream_len, h1_stream_len = len(h0_stream), len(h1_stream)
        
    del h0_stream, h1_stream, data_package, packed_chunk_data, chunks

print(f"\n🧮 [🚰 streaming total] Time:     {np.mean(stream_times):.4f} s ± {np.std(stream_times):.4f} s")
print(f"💾 [🚰 streaming] Peak RAM:     {np.mean(stream_mems):.2f} MiB ± {np.std(stream_mems):.2f} MiB")
print(f"💾 [🚰 streaming] Virtual (VMS): {np.mean(stream_vms):.2f} MiB ± {np.std(stream_vms):.2f} MiB")
print(f" & {m_chunks}& {np.mean(stream_times):.3f}$\\pm${np.std(stream_times):.3f} & {np.mean(stream_mems):.2f}$\\pm${np.std(stream_mems):.2f} \\\\")

# Quick verification verification check against fresh global run
h0_validate, h1_validate = compute_h0_h1_fast(arr_2d)
print(f"\nStreamed Layout -> H0: {h0_stream_len} pairs | H1: {h1_stream_len} pairs")
print(f"Match Success? H0: {'✅' if h0_stream_len == len(h0_validate) else '❌'} | H1: {'✅' if h1_stream_len == len(h1_validate) else '❌'}")

del h0_validate, h1_validate
MemoryUtils.clear_memory()
print("=====================================================================\n")
# watershed gives basin labels directly
# labels = watershed(arr_2d.cpu().numpy(), connectivity=1) # each unique label = one basin = one minimum